# The PlantVillage contrast experiment

Trains the same architecture on laboratory imagery instead of field imagery, so
the difference can be measured rather than asserted.

This notebook is **not** producing a model to ship. It exists to answer one
question: how much of a published accuracy figure survives contact with a real
photograph?

## What to expect

PlantVillage images are single detached leaves, centred, on a uniform grey
background, under even studio light. A model trained here will report something
close to 99% on its own held-out split.

Evaluated on the field photographs collected in Ghana, the same model is
expected to fall a long way, because it has been rewarded for reading a
background that no longer exists rather than the lesion.

The number that goes in the report is the gap between those two figures.

In [ ]:
# Same recipe as 01, pointed at a different dataset. Everything except the data
# is held constant on purpose: if the pipeline changed too, the gap could not be
# attributed to the imagery.
import pathlib
import numpy as np
import tensorflow as tf
import keras

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 32

# PlantVillage carries 38 classes across many crops. Restrict to the cassava
# ones so the comparison is like for like.
DATA_DIR = pathlib.Path("/kaggle/input/plantvillage-dataset/color")
OUT_DIR = pathlib.Path("/kaggle/working")

cassava_dirs = sorted(p for p in DATA_DIR.iterdir() if "cassava" in p.name.lower())
print([p.name for p in cassava_dirs])

In [ ]:
train_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR, labels="inferred", label_mode="int",
    class_names=[p.name for p in cassava_dirs],
    image_size=(IMAGE_SIZE, IMAGE_SIZE), batch_size=BATCH_SIZE,
    validation_split=0.15, subset="training", seed=SEED,
)
val_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR, labels="inferred", label_mode="int",
    class_names=[p.name for p in cassava_dirs],
    image_size=(IMAGE_SIZE, IMAGE_SIZE), batch_size=BATCH_SIZE,
    validation_split=0.15, subset="validation", seed=SEED,
)

Train with the same two-stage schedule as notebook 01, then save as
`plantvillage.keras`. Notebook 03 evaluates both models against the same
Ghanaian field photographs and produces the comparison table.

In [ ]:
# ... same build_model / two-stage fit as 01 ...
# model.save(OUT_DIR / "plantvillage.keras")